<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session05_evaluating_llm_applications_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۵: ارزیابی برنامه‌های مبتنی بر LLM

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session05_evaluating_llm_applications.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json

import pandas as pd
from llm_utils import download_data
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, Optional

reviews = pd.read_csv(download_data("reviews.csv"))
LABELS = ["positive", "negative", "neutral"]


def parse_label(text):
    text = text.strip().lower()
    return next((label for label in LABELS if label in text), "unknown")


def accuracy(classify, data=reviews):
    predictions = pd.Series([classify(t) for t in data.text], index=data.index)
    return (predictions == data.label).mean(), predictions


SYSTEM_EN = """You classify Persian customer reviews of an online electronics store.

Labels:
- positive: the customer is satisfied with the product, delivery or service.
- negative: the customer reports a problem or is dissatisfied.
- neutral: a factual statement or an opinion that is neither clearly positive nor negative.

Answer with exactly one word: positive, negative or neutral."""

EXAMPLES = [
    ("مانیتور بدون حتی یک پیکسل سوخته رسید، رنگ‌هاش هم عالیه.", "positive"),
    ("دو هفته از تاریخ تحویل گذشته و هنوز خبری از سفارشم نیست.", "negative"),
    ("این ماوس با باتری قلمی کار می‌کنه و باتریش داخل جعبه نیست.", "neutral"),
    ("کیبورد خوبیه ولی نور پس‌زمینه‌اش ضعیفه.", "neutral"),
]


def make_classifier(system, examples=()):
    def classify(text):
        messages = [{"role": "system", "content": system}]
        for example, label in examples:
            messages += [{"role": "user", "content": f"Review: {example}"},
                         {"role": "assistant", "content": label}]
        messages.append({"role": "user", "content": f"Review: {text}"})
        return parse_label(chat(client, MODEL, messages=messages, temperature=0))
    return classify


def extract(schema, system, text):
    response = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": text}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()}})
    return schema.model_validate_json(response.choices[0].message.content)


import time

import numpy as np
from sklearn.metrics import f1_score


def run(name, classify, data=reviews):
    start = time.time()
    predictions = [classify(t) for t in data.text]
    return pd.DataFrame({"system": name, "id": data.review_id, "label": data.label,
                         "prediction": predictions, "seconds": (time.time() - start) / len(data)})


def bootstrap_ci(correct, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    correct = np.asarray(correct, dtype=float)
    return np.percentile(rng.choice(correct, size=(n_boot, len(correct))).mean(axis=1), [2.5, 97.5])


def report(results):
    rows = []
    for name, df in results.groupby("system", sort=False):
        correct = (df.prediction == df.label).values
        low, high = bootstrap_ci(correct)
        rows.append({"system": name, "accuracy": correct.mean(), "ci_low": low, "ci_high": high,
                     "macro_f1": f1_score(df.label, df.prediction, labels=LABELS, average="macro", zero_division=0)})
    return pd.DataFrame(rows).set_index("system").round(3)


SYSTEM_FA = """تو نظرات فارسی مشتریان یک فروشگاه اینترنتی لوازم الکترونیکی را دسته‌بندی می‌کنی.
برچسب‌ها:
- positive: مشتری از کالا، ارسال یا خدمات راضی است.
- negative: مشتری مشکلی گزارش می‌کند یا ناراضی است.
- neutral: بیان یک واقعیت یا نظری که به‌وضوح مثبت یا منفی نیست.
فقط یکی از سه کلمهٔ positive، negative یا neutral را بنویس."""

SYSTEMS = {
    "zero-shot EN": make_classifier(SYSTEM_EN),
    "zero-shot FA": make_classifier(SYSTEM_FA),
    "few-shot EN": make_classifier(SYSTEM_EN, EXAMPLES),
}

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

نسخه چهارمی از طبقه‌بند بسازید که Few-shot را با دستورالعمل فارسی ترکیب کند. آن را به جدول مقایسه اضافه کنید و بازه اطمینان را گزارش کنید.

</div>

In [ ]:
SYSTEMS["few-shot FA"] = make_classifier(SYSTEM_FA, EXAMPLES)
results = pd.concat([run(name, fn) for name, fn in SYSTEMS.items()], ignore_index=True)
report(results)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

ستون‌های `ci_low` و `ci_high` بازهٔ اطمینان ۹۵ درصدی دقت‌اند. با ۴۸ نمونه، پهنای این بازه معمولاً بیش از ۲۰ درصد است. پس اگر بازه‌های دو سیستم هم‌پوشانی زیادی دارند، نمی‌توان گفت کدام بهتر است.

برای انتخاب بین سیستم‌هایی که از نظر آماری هم‌ترازند، معیارهای دیگر را ببینید: تعداد توکن ورودی (Few-shot گران‌تر است)، زمان پاسخ، و رفتار روی کلاس مهم‌تر، یعنی Recall کلاس منفی.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

ده نظر جدید و دشوار بنویسید، مثلاً نظرات طعنه‌آمیز یا نظراتی که هم مثبت و هم منفی‌اند، و به مجموعه آزمون اضافه کنید. رتبه پرامپت‌ها تغییر می‌کند؟

</div>

In [ ]:
hard = pd.DataFrame([
    ("وای چه گوشی فوق‌العاده‌ای! فقط سه بار تا حالا خودبه‌خود خاموش شده.", "negative"),
    ("آفرین به این سرعت ارسال، فقط دو هفته طول کشید!", "negative"),
    ("کیفیت ساخت عالیه ولی نرم‌افزارش پر از باگه و پشتیبانی هم جواب نمیده.", "negative"),
    ("صفحه‌اش یه کم کم‌نوره ولی با این قیمت واقعاً می‌ارزه، راضیم.", "positive"),
    ("نسبت به گوشی قبلیم بهتره.", "positive"),
    ("بد نیست.", "neutral"),
    ("برای هدیه گرفتم، هنوز بازش نکرده.", "neutral"),
    ("انتظار بیشتری داشتم.", "negative"),
    ("همون چیزی بود که توی عکس بود، نه کمتر نه بیشتر.", "neutral"),
    ("اگه پشتیبانی بهتر بود پنج ستاره می‌دادم، فعلاً چهار.", "positive"),
], columns=["text", "label"])
hard["review_id"] = [f"H{i:02d}" for i in range(1, len(hard) + 1)]

hard_results = pd.concat([run(name, fn, hard) for name, fn in SYSTEMS.items()], ignore_index=True)
print("hard set only:")
print(report(hard_results)[["accuracy", "macro_f1"]])
combined = pd.concat([results, hard_results], ignore_index=True)
print("\noriginal + hard:")
print(report(combined)[["accuracy", "macro_f1"]])

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

نظرات طعنه‌آمیز («آفرین به این سرعت ارسال، فقط دو هفته!») و نظرات با جنبه‌های مثبت و منفی هم برای مدل و هم برای انسان دشوارند. برچسب بعضی از آن‌ها قابل بحث است. مثلاً «اگه پشتیبانی بهتر بود پنج ستاره می‌دادم، فعلاً چهار» را بعضی‌ها خنثی می‌دانند. بخشی از کار ساخت مجموعهٔ آزمون، نوشتن **راهنمای برچسب‌گذاری** برای همین موارد است.

ارزش اصلی این تمرین آن است که رتبهٔ سیستم‌ها روی دادهٔ آسان ممکن است با رتبه روی دادهٔ دشوار فرق کند. مجموعهٔ آزمونی که فقط نمونه‌های ساده دارد، تفاوت واقعی سیستم‌ها را پنهان می‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

پنج پاسخ از جدول `grades` را خودتان با همان معیار نمره بدهید. توافق شما با داور چقدر است؟

</div>

In [ ]:
docs_qa = [json.loads(line) for line in open(download_data("rag_eval.jsonl"), encoding="utf-8")][:5]
from llm_utils import load_store_docs
docs = load_store_docs()


class Grade(BaseModel):
    reasoning: str
    score: Literal[1, 2, 3, 4, 5]


RUBRIC = """Compare the ANSWER with the REFERENCE and grade factual correctness and completeness:
5 = all facts present and correct; 4 = correct, minor detail missing; 3 = partly correct;
2 = mostly wrong; 1 = wrong or irrelevant. Write a short reasoning first, then the score."""

rows = []
for item in docs_qa:
    answer = chat(client, MODEL, f"<document>\n{docs[item['source']]}\n</document>\n\nپرسش: {item['question']}",
                  system="فقط بر اساس سند و در یک جمله پاسخ بده.", temperature=0)
    grade = extract(Grade, RUBRIC, f"QUESTION: {item['question']}\nREFERENCE: {item['answer']}\nANSWER: {answer}")
    rows.append({"question": item["question"], "reference": item["answer"], "answer": answer, "judge": grade.score})
grades = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
grades

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

جدول بالا را بخوانید و نمرهٔ خودتان را با همان معیار در فهرست زیر بنویسید. مقدارهای فعلی فقط نمونه‌اند.

</div>

In [ ]:
my_scores = [5, 4, 5, 3, 5]          # replace with your own grades, in the same order as the table

grades["mine"] = my_scores
exact = (grades.mine == grades.judge).mean()
within_one = ((grades.mine - grades.judge).abs() <= 1).mean()
print(f"exact agreement: {exact:.0%}   agreement within one point: {within_one:.0%}")
grades[["question", "judge", "mine"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برای نمره‌های ۱ تا ۵، توافق کامل سخت‌گیرانه است. توافق «با اختلاف حداکثر یک نمره» معیار رایج‌تری است. اگر داور و شما روی نمونه‌ای اختلاف زیادی دارید، دلیل داور (`reasoning`) را بخوانید. معمولاً یکی از این‌هاست: معیار مبهم است (مثلاً «جزئیات جزئی» یعنی چه؟)، پاسخ مرجع ناقص است، یا داور شرطی را نادیده گرفته است. با پنج نمونه نمی‌توان دربارهٔ داور قضاوت کرد؛ در عمل دست‌کم ۳۰ تا ۵۰ نمونه را دستی بررسی می‌کنند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

برای سنجش **سوگیری طول**، پاسخ `A_brief` را با افزودن یک جمله کلی و بی‌اطلاعات («امیدواریم از خرید خود لذت ببرید.») طولانی کنید و داوری جفتی را تکرار کنید. نتیجه چه تغییری کرد؟

</div>

In [ ]:
ASSISTANTS = {
    "A_brief": "فقط بر اساس سند و در حداکثر یک جمله پاسخ بده.",
    "B_helpful": "فقط بر اساس سند پاسخ بده. پاسخ را کامل ولی کوتاه بنویس و شرط‌ها و استثناها را هم بگو.",
}
PADDING = " امیدواریم از خرید خود لذت ببرید."
PAIRWISE = ("You compare two answers of a Persian customer-support assistant to the same question. "
            "Prefer the answer that is correct, complete and clear for a customer. "
            "Reply with exactly one character: 1 if the first answer is better, 2 if the second is better.")


def prefer(question, first, second):
    out = chat(client, MODEL, f"QUESTION: {question}\n\nANSWER 1: {first}\n\nANSWER 2: {second}",
               system=PAIRWISE, temperature=0, max_tokens=3).strip()
    return 1 if out.startswith("1") else 2


def a_win_rate(padding=""):
    wins = []
    for item in docs_qa:
        context = f"<document>\n{docs[item['source']]}\n</document>\n\nپرسش: {item['question']}"
        a = chat(client, MODEL, context, system=ASSISTANTS["A_brief"], temperature=0) + padding
        b = chat(client, MODEL, context, system=ASSISTANTS["B_helpful"], temperature=0)
        # both orders; a win counts only when the verdict is consistent
        wins.append(prefer(item["question"], a, b) == 1 and prefer(item["question"], b, a) == 2)
    return np.mean(wins)


print(f"A wins without padding: {a_win_rate():.0%}")
print(f"A wins with padding:    {a_win_rate(PADDING):.0%}")

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

جملهٔ اضافه هیچ اطلاعاتی ندارد، پس داور منصف نباید نظرش را عوض کند. اگر نرخ برد A با این جمله بالا رفته، داور **سوگیری طول یا لحن** دارد. با پنج پرسش، حتی یک تغییر ۲۰ درصد اختلاف ایجاد می‌کند. برای نتیجهٔ قابل اعتماد، آزمایش را روی همهٔ ۳۰ پرسش اجرا کنید.

دقت کنید که فقط داوری‌های **سازگار در هر دو ترتیب** را برد حساب کردیم. این کار اثر سوگیری جایگاه را از اثر طول جدا می‌کند. اگر نمی‌کردیم، تغییر نتیجه ممکن بود فقط از جابه‌جایی تصادفی ترتیب باشد.

</div>